# 01 — Base común (limpieza e imputación única)

easyMoney · TFM Nuclio Digital School

Siguiendo el feedback de la tutora, **toda la limpieza e imputación se hace una sola vez, aquí**. De esta base salen las ramas de modelado:

```
CSV en bruto ──► 01_base_comun ──► df_panel_limpio.parquet ──┬─► 02  Tarea 2 (propensión)
                                  df_sales_limpio.parquet  ──┼─► 03  Tarea 3 (segmentación)
                                                             └─► 04  Tarea 4 (business case)
```

**Qué hace este notebook:** limpiar, imputar, unir y guardar.

**Qué NO hace:** feature engineering. Los `_lag1`, `num_products`, `mes` o `antigüedad` son decisiones de cada tarea y se construyen dentro de su rama. Así ninguna tarea tiene que *deshacer* lo que hizo otra.


In [1]:
import os
import numpy as np
import pandas as pd

RUTA_RAW = '../data/raw/'
RUTA_OUT = '../data/processed/'
os.makedirs(RUTA_OUT, exist_ok=True)

df_sal = pd.read_csv(RUTA_RAW + 'sales.csv', index_col=0)
df_cs  = pd.read_csv(RUTA_RAW + 'customer_sociodemographics.csv', index_col=0)
df_cca = pd.read_csv(RUTA_RAW + 'customer_commercial_activity.csv', index_col=0)
df_cp  = pd.read_csv(RUTA_RAW + 'customer_products.csv', index_col=0)

for nombre, d in [('sales', df_sal), ('sociodemographics', df_cs),
                  ('commercial_activity', df_cca), ('products', df_cp)]:
    print(f'{nombre:20s} {d.shape}')

sales                (240773, 5)
sociodemographics    (5962924, 8)
commercial_activity  (5962924, 6)
products             (5962924, 17)


## 1. Productos (`customer_products`)

Solo `payroll` y `pension_plan` traen nulos (61 filas cada uno), pero aplicamos el criterio a las 15 columnas por si acaso.

In [2]:
PRODUCT_COLS = ['short_term_deposit', 'loans', 'mortgage', 'funds', 'securities',
                'long_term_deposit', 'em_account_pp', 'credit_card', 'payroll',
                'pension_plan', 'payroll_account', 'emc_account', 'debit_card',
                'em_account_p', 'em_acount']

print('Nulos antes:', df_cp[PRODUCT_COLS].isnull().sum()[lambda s: s > 0].to_dict())
df_cp[PRODUCT_COLS] = df_cp[PRODUCT_COLS].fillna(0).astype(int)
print('Nulos después:', df_cp[PRODUCT_COLS].isnull().sum().sum())

Nulos antes: {'payroll': 61, 'pension_plan': 61}
Nulos después: 0


## 2. Fallecidos fuera

Se quitan los **meses** en los que el cliente figura como fallecido. Al hacer después un `inner` con esta tabla, esos meses desaparecen también del panel y, por tanto, de las tareas 2, 3 y 4.

In [3]:
print(df_cs['deceased'].value_counts(dropna=False))
filas_antes = len(df_cs)
df_cs = df_cs[df_cs['deceased'] == 'N'].drop(columns=['deceased'])
print(f'\nFilas eliminadas por fallecimiento: {filas_antes - len(df_cs)}')

deceased
N    5961849
S       1075
Name: count, dtype: int64

Filas eliminadas por fallecimiento: 1075


## 3. Construcción del panel cliente-mes

Unimos las tres tablas por `pk_cid` + `pk_partition`:
- `inner` con sociodemográficos: así heredamos el filtro de fallecidos.
- `left` con actividad comercial: no hemos filtrado nada en esa tabla.

Ordenamos por cliente y fecha. Cualquier `ffill`, `bfill` o `shift` posterior depende de este orden; si no estuviera ordenado, saldría mal sin dar ningún error.

In [4]:
for d in (df_cp, df_cs, df_cca):
    d['pk_partition'] = pd.to_datetime(d['pk_partition'], format='%Y-%m')
df_cca['entry_date'] = pd.to_datetime(df_cca['entry_date'], format='%Y-%m')

df_panel = (df_cp
            .merge(df_cs, on=['pk_cid', 'pk_partition'], how='inner')
            .merge(df_cca, on=['pk_cid', 'pk_partition'], how='left')
            .sort_values(['pk_cid', 'pk_partition'])
            .reset_index(drop=True))

print('Forma del panel:', df_panel.shape)
print('Clientes únicos:', df_panel['pk_cid'].nunique())
print('Meses:', df_panel['pk_partition'].min().strftime('%Y-%m'), '->',
      df_panel['pk_partition'].max().strftime('%Y-%m'))
print('\nNulos tras el merge:')
print(df_panel.isnull().sum()[lambda s: s > 0])

Forma del panel: (5961849, 26)
Clientes únicos: 456318
Meses: 2018-01 -> 2019-05

Nulos tras el merge:
region_code         2259
gender                25
salary           1540876
entry_channel     133033
segment           133944
dtype: int64


## 4. Imputación con la historia del propio cliente

`gender`, `region_code`, `entry_channel` y `segment` son atributos del cliente: si falta en un mes, lo más fiable es el valor que ese mismo cliente tiene en otros meses.

- Primero `ffill`: el último valor conocido.
- Después `bfill`: para los nulos de los primeros meses del cliente.

**Sobre el `bfill` y el leakage:** rellenar un mes con un valor de un mes posterior usa información futura. Aquí lo aceptamos porque son atributos asignados en el alta (canal de entrada, segmento, provincia, género), no comportamiento. No dicen nada sobre si el cliente comprará un producto. Lo dejamos documentado por si lo pregunta el tribunal.

Lo que siga a nulo después de esto (clientes que **nunca** tienen el dato) pasa a los criterios de la sección siguiente.

In [5]:
COLS_HISTORIA = ['gender', 'region_code', 'entry_channel', 'segment']

nulos_antes = df_panel[COLS_HISTORIA].isnull().sum()
df_panel[COLS_HISTORIA] = df_panel.groupby('pk_cid')[COLS_HISTORIA].ffill()
df_panel[COLS_HISTORIA] = df_panel.groupby('pk_cid')[COLS_HISTORIA].bfill()
nulos_despues = df_panel[COLS_HISTORIA].isnull().sum()

print(pd.DataFrame({'nulos_antes': nulos_antes,
                    'resueltos_con_historia': nulos_antes - nulos_despues,
                    'quedan': nulos_despues}))

               nulos_antes  resueltos_con_historia  quedan
gender                  25                       0      25
region_code           2259                     126    2133
entry_channel       133033                  123698    9335
segment             133944                  124876    9068


## 5. Lo que la historia no resuelve

- **`gender`:** moda global. Afecta a muy pocas filas.
- **`region_code`:** moda de España para clientes con `country_id == 'ES'`, `0` para extranjeros. Por eso `country_id` **no** se borra antes de imputar.
- **`entry_channel` / `segment`:** imputación cruzada. Si falta el canal pero hay segmento, se usa la moda del canal dentro de ese segmento, y al revés. Si faltan los dos, `"Desconocido"`.

In [6]:
# gender
df_panel['gender'] = df_panel['gender'].fillna(df_panel['gender'].mode()[0])

# region_code condicionado a country_id
moda_region_es = df_panel.loc[df_panel['country_id'] == 'ES', 'region_code'].mode()[0]
mask_es  = df_panel['region_code'].isnull() & (df_panel['country_id'] == 'ES')
mask_ext = df_panel['region_code'].isnull() & (df_panel['country_id'] != 'ES')
df_panel.loc[mask_es,  'region_code'] = moda_region_es
df_panel.loc[mask_ext, 'region_code'] = 0
df_panel['region_code'] = df_panel['region_code'].astype(int)
print(f'region_code -> moda ES ({moda_region_es:.0f}): {mask_es.sum()} | extranjero (0): {mask_ext.sum()}')

# entry_channel / segment: imputación cruzada
def moda_segura(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

mask_canal = df_panel['entry_channel'].isnull() & df_panel['segment'].notnull()
mask_seg   = df_panel['segment'].isnull() & df_panel['entry_channel'].notnull()

moda_canal_por_seg = df_panel.groupby('segment')['entry_channel'].agg(moda_segura)
moda_seg_por_canal = df_panel.groupby('entry_channel')['segment'].agg(moda_segura)

df_panel.loc[mask_canal, 'entry_channel'] = df_panel.loc[mask_canal, 'segment'].map(moda_canal_por_seg)
df_panel.loc[mask_seg,   'segment']       = df_panel.loc[mask_seg, 'entry_channel'].map(moda_seg_por_canal)

mask_ambos = df_panel['entry_channel'].isnull() | df_panel['segment'].isnull()
df_panel['entry_channel'] = df_panel['entry_channel'].fillna('Desconocido')
df_panel['segment'] = df_panel['segment'].fillna('Desconocido')
print(f'Canal por segment: {mask_canal.sum()} | Segment por canal: {mask_seg.sum()} | Desconocido: {mask_ambos.sum()}')

# active_customer
df_panel['active_customer'] = df_panel['active_customer'].fillna(0).astype(int)

region_code -> moda ES (28): 0 | extranjero (0): 2133
Canal por segment: 612 | Segment por canal: 345 | Desconocido: 8724


## 6. `salary`: imputación

`salary` es la variable con más nulos. Se hace en cuatro pasos, del más fiable al menos fiable:

1. **Marcar los nulos originales** con `salary_imputado` (1 = estimado, 0 = real), antes de tocar nada.
2. **Historia del propio cliente:** si le falta en un mes pero lo tiene en otro, se copia su propio valor.
3. **Ver de qué depende el salario** para elegir con qué variables agrupar.
4. **Mediana de clientes parecidos** para los que no tienen salario en ningún mes.

Se usa la **mediana** y no la media porque `salary` tiene valores extremos que inflarían la media.

In [7]:
# Paso 1: marcar los nulos originales (antes de imputar nada)
df_panel['salary_imputado'] = df_panel['salary'].isnull().astype(int)
print(f"Salary nulo original: {df_panel['salary_imputado'].sum():,} filas "
      f"({df_panel['salary_imputado'].mean():.1%} del panel)")

# Paso 2: historia del propio cliente (hacia delante y hacia atrás)
df_panel = df_panel.sort_values(['pk_cid', 'pk_partition'])
df_panel['salary'] = df_panel.groupby('pk_cid')['salary'].transform(lambda s: s.ffill().bfill())

pendientes = df_panel['salary'].isnull().sum()
print(f"Recuperados con la historia del cliente: {df_panel['salary_imputado'].sum() - pendientes:,}")
print(f"Siguen sin salary (el cliente no lo tiene en ningún mes): {pendientes:,} filas "
      f"| {df_panel.loc[df_panel['salary'].isnull(), 'pk_cid'].nunique():,} clientes")

Salary nulo original: 1,540,876 filas (25.8% del panel)
Recuperados con la historia del cliente: 621
Siguen sin salary (el cliente no lo tiene en ningún mes): 1,540,255 filas | 158,752 clientes


### Paso 3: ¿de qué depende el salario?

Para cada variable candidata se mira cuánto cambia la mediana del salario entre sus categorías, y qué parte de la variación del salario explica (**η²**, de 0 a 1: cuanto más alto, más explica la variable).

Se calcula sobre el **logaritmo** del salario para que los sueldos extremos no dominen el cálculo, y solo con los valores reales (`salary_imputado == 0`).

In [8]:
df_panel['tramo_edad'] = pd.cut(df_panel['age'], bins=[0, 25, 35, 45, 55, 65, 120],
                                labels=['<25', '25-34', '35-44', '45-54', '55-64', '65+'])

reales = df_panel[df_panel['salary_imputado'] == 0].copy()
reales['log_salary'] = np.log1p(reales['salary'])
var_total = ((reales['log_salary'] - reales['log_salary'].mean()) ** 2).sum()

filas = []
for col in ['region_code', 'segment', 'tramo_edad', 'gender', 'entry_channel']:
    g = reales.groupby(col, observed=True)['log_salary'].agg(['mean', 'count'])
    var_entre = (g['count'] * (g['mean'] - reales['log_salary'].mean()) ** 2).sum()
    med = reales.groupby(col, observed=True)['salary'].median()
    filas.append({'variable': col, 'categorias': len(g),
                  'mediana_min': round(med.min()), 'mediana_max': round(med.max()),
                  'eta2': round(var_entre / var_total, 3)})

relacion_salary = pd.DataFrame(filas).sort_values('eta2', ascending=False)
print(relacion_salary.to_string(index=False))

     variable  categorias  mediana_min  mediana_max  eta2
  region_code          52        53029       126179 0.189
entry_channel          61        32627       402884 0.007
      segment           4        87851       121099 0.004
   tramo_edad           6        85370       102747 0.004
       gender           2        87201        89909 0.001


### Paso 4: mediana de clientes parecidos

A cada fila sin salario se le pone la mediana de los clientes con su misma combinación de variables (`VARIABLES_SALARY`, de más a menos importante).

Si su combinación tiene menos de `MIN_GRUPO` clientes con salario real, la mediana no es fiable y se **baja un nivel** quitando la última variable de la lista, y así hasta llegar a la mediana global. Por ejemplo, con `['region_code', 'segment', 'tramo_edad']`:

1. región × segmento × tramo de edad
2. si no hay suficientes → región × segmento
3. si no → región
4. si no → mediana global

Las medianas se calculan solo con salarios **reales**, nunca con valores ya imputados.

In [9]:
VARIABLES_SALARY = ['region_code', 'segment', 'tramo_edad']   # revisar con la tabla del paso 3
MIN_GRUPO = 30                                                 # clientes mínimos para fiarse de una mediana

# Una fila por cliente con salario real (para que un cliente con 17 meses no pese 17 veces)
base_medianas = (df_panel[df_panel['salary_imputado'] == 0]
                 .groupby('pk_cid').last()[VARIABLES_SALARY + ['salary']])

faltan = df_panel['salary'].isnull()
origen = pd.Series('real o historia', index=df_panel.index)

for n in range(len(VARIABLES_SALARY), 0, -1):
    claves = VARIABLES_SALARY[:n]
    stats = base_medianas.groupby(claves, observed=True)['salary'].agg(['median', 'count'])
    stats = stats[stats['count'] >= MIN_GRUPO]['median'].rename('mediana_grupo').reset_index()

    pendientes = (df_panel.loc[faltan, claves].rename_axis('fila').reset_index()
                  .merge(stats, on=claves, how='left').set_index('fila'))
    rellenar = pendientes['mediana_grupo'].dropna()
    df_panel.loc[rellenar.index, 'salary'] = rellenar
    origen.loc[rellenar.index] = ' × '.join(claves)
    faltan = df_panel['salary'].isnull()

mediana_global = base_medianas['salary'].median()
origen.loc[faltan] = 'mediana global'
df_panel['salary'] = df_panel['salary'].fillna(mediana_global)

print('Cómo se ha imputado cada fila con salary nulo original:')
print(origen[df_panel['salary_imputado'] == 1].value_counts().to_string())
print(f"\nNulos en salary tras imputar: {df_panel['salary'].isnull().sum()}")

df_panel = df_panel.drop(columns=['tramo_edad'])

Cómo se ha imputado cada fila con salary nulo original:
region_code × segment × tramo_edad    1415930
mediana global                         100207
region_code × segment                   16305
region_code                              7813
real o historia                           621

Nulos en salary tras imputar: 0


In [10]:
# Comprobación: la distribución de salary no debe deformarse al imputar
comparacion = pd.DataFrame({
    'reales':    df_panel.loc[df_panel['salary_imputado'] == 0, 'salary'].describe(),
    'imputados': df_panel.loc[df_panel['salary_imputado'] == 1, 'salary'].describe(),
}).round(0)
print(comparacion)

           reales  imputados
count   4420973.0  1540876.0
mean     115581.0    91920.0
std      200018.0    24959.0
min        1203.0    27669.0
25%       61414.0    70120.0
50%       88438.0    87993.0
75%      131308.0   108593.0
max    28894396.0   470033.0


**Nota.** Las medianas se calculan con todos los meses (enero 2018 – mayo 2019). Es una decisión de limpieza de la base común: el valor imputado de un cliente es una característica estable (su salario), no un dato que cambie mes a mes.

## 7. Ventas (`sales`)

1. `month_sale` se pasa a fecha mensual, con el mismo formato que `pk_partition`.
2. Nos quedamos solo con las ventas de cliente-mes que existen en el panel. Así las ventas de meses de clientes fallecidos también quedan fuera.
3. Se guardan dos salidas:
   - **Ventas limpias** (una fila por venta) para la Tarea 4, que necesita el margen por producto.
   - **Agregado por cliente-mes** (`n_compras_mes`, `margen_mes`) unido al panel, para la Tarea 3.

In [11]:
df_sal['pk_partition'] = pd.to_datetime(df_sal['month_sale']).dt.to_period('M').dt.to_timestamp()
df_sal = df_sal.rename(columns={'cid': 'pk_cid'}).drop(columns=['month_sale'])

claves_panel = df_panel[['pk_cid', 'pk_partition']]
ventas_antes = len(df_sal)
df_sales_limpio = df_sal.merge(claves_panel, on=['pk_cid', 'pk_partition'], how='inner')
print(f'Ventas totales: {ventas_antes} | en el panel: {len(df_sales_limpio)} | '
      f'descartadas (cliente-mes fuera del panel): {ventas_antes - len(df_sales_limpio)}')

ventas_mes = (df_sales_limpio
              .groupby(['pk_cid', 'pk_partition'])
              .agg(n_compras_mes=('pk_sale', 'count'), margen_mes=('net_margin', 'sum'))
              .reset_index())

df_panel = df_panel.merge(ventas_mes, on=['pk_cid', 'pk_partition'], how='left')
df_panel[['n_compras_mes', 'margen_mes']] = df_panel[['n_compras_mes', 'margen_mes']].fillna(0)
df_panel['n_compras_mes'] = df_panel['n_compras_mes'].astype(int)

# Comprobación: el agregado del panel cuadra con las ventas limpias
assert df_panel['n_compras_mes'].sum() == len(df_sales_limpio)
assert np.isclose(df_panel['margen_mes'].sum(), df_sales_limpio['net_margin'].sum())
print('Agregado de ventas cuadra con las ventas limpias.')

Ventas totales: 240773 | en el panel: 240752 | descartadas (cliente-mes fuera del panel): 21
Agregado de ventas cuadra con las ventas limpias.


## 8. Verificación final y guardado

In [12]:
df_panel = df_panel.sort_values(['pk_cid', 'pk_partition']).reset_index(drop=True)

nulos = df_panel.isnull().sum()
print('Columnas con nulos (no debería aparecer ninguna):')
print(nulos[nulos > 0] if nulos.any() else 'Ninguna')
assert not nulos.any(), 'Hay nulos inesperados'
assert not df_panel.duplicated(['pk_cid', 'pk_partition']).any(), 'Hay cliente-mes duplicados'

print('\nForma final:', df_panel.shape)
print('Columnas:', list(df_panel.columns))

Columnas con nulos (no debería aparecer ninguna):
Ninguna

Forma final: (5961849, 29)
Columnas: ['pk_cid', 'pk_partition', 'short_term_deposit', 'loans', 'mortgage', 'funds', 'securities', 'long_term_deposit', 'em_account_pp', 'credit_card', 'payroll', 'pension_plan', 'payroll_account', 'emc_account', 'debit_card', 'em_account_p', 'em_acount', 'country_id', 'region_code', 'gender', 'age', 'salary', 'entry_date', 'entry_channel', 'active_customer', 'segment', 'salary_imputado', 'n_compras_mes', 'margen_mes']


In [13]:
df_panel.to_parquet(RUTA_OUT + 'df_panel_limpio.parquet', engine='pyarrow', index=False)
df_sales_limpio.to_parquet(RUTA_OUT + 'df_sales_limpio.parquet', engine='pyarrow', index=False)
print('Guardado:')
print(' -', RUTA_OUT + 'df_panel_limpio.parquet', df_panel.shape)
print(' -', RUTA_OUT + 'df_sales_limpio.parquet', df_sales_limpio.shape)

Guardado:
 - ../data/processed/df_panel_limpio.parquet (5961849, 29)
 - ../data/processed/df_sales_limpio.parquet (240752, 5)


## Cómo lo usa cada tarea

- **`02_tarea2_propension`:** crea `num_products`, `mes`, `antigüedad`, los `_lag1` y los targets, y compara 4 modelos para `pension_plan` y `em_acount`.
- **`03_tarea3_segmentacion`:** foto de clientes + histórico de ventas.
- **Tarea 4:** usa `df_sales_limpio.parquet` para el margen por producto, más los scorings de la Tarea 2 y los grupos de la Tarea 3. Los tres salen del mismo universo de clientes.